# Can an SGD-magnitude graft lower AdamW’s Shakespeare loss?

Compare native AdamW, SGD with momentum, and AdamW with the MNIST magnitude graft on the same character-level nanoGPT. Each arm uses the same five initializations and minibatch streams. The primary loss comparison is the paired difference in held-out cross-entropy at 20,000 updates. WeightWatcher measures the layer spectra of the trained models.

Run the cells in order using the Python environment from the [setup instructions](README.md). Opening the notebook starts no training.

## Set the device and paths

Use `cpu` on Linux or a Mac, `mps` on a supported Apple Silicon Mac, or `tpu` in a TorchTPU kernel on a TPU VM. The defaults run all 15 fits on CPU. To stage one fit, set `ARM = "graft"`, `SEED = 1340`, and `UNTIL_STEP = 500`. Set all three back to `None` to resume that fit and complete the comparison. Use a separate output directory for each backend.

In [ ]:
from pathlib import Path
import hashlib
import subprocess
import sys

DEVICE = "cpu"
ARM = None          # None, "adamw", "sgdm", or "graft"
SEED = None         # None or an integer from 1340 through 1344
UNTIL_STEP = None   # None for the full endpoint, or e.g. 500 for a staged run

# If Jupyter started outside the repository, set START to its absolute path.
START = Path.cwd()
ROOT = next((p for p in (START, *START.parents)
             if (p / "short_test_adamw_graft" / "run.py").is_file()
             and (p / "short_test" / "download_data.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Set START to the rg_optimizers checkout and rerun this cell.")
EXPERIMENT = ROOT / "short_test_adamw_graft"
DATA = EXPERIMENT / "data" / "input.txt"
OUTPUT = EXPERIMENT / "runs" / DEVICE

def run_script(script, *args):
    command = [sys.executable, "-u", str(script), *map(str, args)]
    subprocess.run(command, cwd=ROOT, check=True)

print(f"Kernel: {sys.executable}\nDevice: {DEVICE}\nCorpus: {DATA}\nOutput: {OUTPUT}")

## Verify the corpus

The downloader fetches Tiny Shakespeare once. Every run checks the same corpus checksum, including when you point `DATA` at an existing copy.

In [ ]:
if not DATA.exists():
    run_script(ROOT / "short_test" / "download_data.py", "--output", DATA)
expected_sha = "86c4e6aa9db7c042ec79f339dcb96d42b0075e16b8fc2e86bf0ca57e2dc565ed"
if hashlib.sha256(DATA.read_bytes()).hexdigest() != expected_sha:
    raise ValueError(f"Tiny Shakespeare checksum mismatch at {DATA}")
print(f"Verified {DATA.stat().st_size:,} bytes")

## Check the selected backend

Run this before training, particularly on a new Mac or TPU installation.

In [ ]:
runner = EXPERIMENT / "run.py"
common_args = ["--device", DEVICE, "--data", DATA, "--output", OUTPUT]
run_script(runner, "preflight", *common_args)

## Train or resume

Running this cell with the defaults requests 15 fits of 20,000 updates each. For an initial timing measurement, use the one-fit, 500-step settings above. Keep the kernel attached while this cell runs or use the equivalent shell command in the README.

In [ ]:
selection = []
if ARM is not None:
    selection.extend(["--arm", ARM])
if SEED is not None:
    selection.extend(["--seed", SEED])
if UNTIL_STEP is not None:
    selection.extend(["--until-step", UNTIL_STEP])
run_script(runner, "run", *common_args, *selection)

## Compare the outcomes

Use endpoint cross-entropy in nats per character. A negative graft-minus-AdamW difference favors the graft. Inspect the five paired seeds alongside the aggregate. Per-matrix rows identify which layers move toward or away from α = 2. Partial checkpoints have no endpoint results.

In [ ]:
import pandas as pd
from IPython.display import display

loss = "test_loss_nats_per_character"
alpha = "confidence_tail_weighted_alpha"
finished = sorted(OUTPUT.glob("*/seed_*/result.json"))
print(f"{len(finished)} of 15 runs have final results")
if finished:
    run_script(EXPERIMENT / "report.py", "--output", OUTPUT)
    per_seed = pd.read_csv(OUTPUT / "per_seed.csv")
    arm_means = pd.read_csv(OUTPUT / "arm_means.csv")
    paired = pd.read_csv(OUTPUT / "paired_effects.csv")
    per_matrix = pd.read_csv(OUTPUT / "per_matrix.csv")

    end_seeds = per_seed.loc[per_seed["checkpoint"].eq("endpoint")]
    print("Test loss by matched seed · nats per character")
    display(end_seeds.pivot(index="seed", columns="arm", values=loss))

    end_means = arm_means.loc[arm_means["checkpoint"].eq("endpoint")
                              & arm_means["metric"].isin([loss, alpha])]
    print("Endpoint means across available seeds")
    display(end_means[["arm", "metric", "mean", "n", "expected_n"]])

    effects = paired.loc[paired["checkpoint"].eq("endpoint") & paired["level"].eq("mean")
                         & paired["metric"].isin([loss, alpha])]
    print("Graft minus each control · paired differences with 95% intervals")
    display(effects[["baseline", "metric", "graft_minus_baseline", "ci95_low", "ci95_high", "n", "status"]])
else:
    print("Complete a 20,000-step fit to populate the endpoint tables.")

## Inspect endpoint layer alphas

Choose one of the paired seeds to see its raw WeightWatcher fits. The full table in `per_matrix.csv` contains the fit status, KS distance, tail count, and confidence weight for each matrix and checkpoint.

In [ ]:
INSPECT_SEED = 1340
if finished:
    layer_rows = per_matrix.loc[per_matrix["seed"].eq(INSPECT_SEED)
                                & per_matrix["checkpoints"].str.contains("endpoint", na=False)]
    if layer_rows.empty:
        print(f"No endpoint matrix fits available for seed {INSPECT_SEED}")
    else:
        display(layer_rows[["arm", "matrix", "fit_status", "fit_alpha", "fit_D",
                            "fit_num_pl_spikes", "confidence_weight"]].sort_values(["matrix", "arm"]))